# 第13課 - 代理記憶


## 設定

此筆記本展示如何使用 **Microsoft Agent Framework** (MAF) 建立具有 **持久記憶** 的旅遊預訂代理。

您將學習不同類型的代理記憶 — 工作記憶、短期記憶和長期記憶 — 如何影響代理在對話中保留和使用資訊的方式。

**先決條件：**
- 已部署聊天模型（例如 `gpt-4o-mini`）的 Azure AI Foundry 專案。
- 已使用 Azure CLI 登入 — 在終端機執行 `az login`。
- `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## 代理記憶類型

AI 代理可以利用不同種類的記憶，每種記憶具備不同的用途：

### 工作記憶
對話主線本身 — 在單一會話中交換的訊息。代理可以回頭參考同一主線中的早期訊息以維持連貫性。在 MAF 中，這是透過 **`agent.create_session()`** 創建，並回傳一個 `AgentSession`。

### 短期記憶
持續於任務或會話期間的資訊，但不會永久儲存。例如，代理在多回合的規劃對話中累積的事實，並用來生成最終行程。

### 長期記憶
跨會話持續存在的偏好與事實。回訪的使用者不必重複他們的飲食限制或旅遊風格。長期記憶通常由外部儲存支援 — 資料庫、檔案或向量索引 — 並透過工具呈現給代理。


## 使用會話的工作記憶

最簡單的記憶形式是會話階段。當你將相同的會話（透過 `agent.create_session()` 建立）傳遞給連續的 `agent.run()` 呼叫時，代理人可以看到該會話的完整歷史，並且可以回憶早先的細節。

讓我們創建一個旅遊代理人並示範工作記憶。


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

代理正確地回想起預算是因為兩則訊息共享相同的會話。這是**工作記憶**——它僅存在於會話期間。 

### 新的對話串會發生什麼事？

如果我們建立一個**新的**會話，代理將不會記得先前的對話：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 長期記憶模式

要記住使用者偏好 **跨越多個會話**，我們需要一個存在於對話線程之外的持久存儲。代理通過 **工具** 訪問此存儲 — 也就是它可以調用以保存和檢索資訊的函數。

以下我們實作一個簡單的記憶體內偏好存儲（在生產環境中您會用資料庫或向量索引來支援），並將其暴露為代理可使用的工具。

### 架構
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 情境 1 — 首次使用者預訂週年旅行

Sarah 第一次造訪。客服人員應透過工具儲存她的喜好，並利用這些喜好來推薦飯店。


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 情境二 — Sarah 幾週後回來

Sarah 開始一個**全新的對話串**（模擬新的會話）。工作記憶是空的，但長期偏好儲存仍然有她的資訊。代理程式應該從中檢索並用來個人化推薦。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 摘要

在本課程中，您學到了三種類型的代理記憶體以及如何使用 Microsoft 代理框架來實現它們：

| 記憶體類型 | MAF 機制 | 壽命 |
|---|---|---|
| **工作記憶** | `agent.create_session()` | 單次對話 |
| **短期記憶** | 線程內累積的上下文 | 單一任務 / 會話 |
| **長期記憶** | 透過 `@tool` 函數存取的外部存儲 | 跨會話 |

### 主要重點
1. **`agent.create_session()`** 提供工作記憶 — 代理在會話內能看到完整的對話歷史。
2. **新會話會失去上下文** — 若無長期記憶，代理無法回憶過去對話。
3. **`@tool` 函數彌補了這個差距** — 它們讓代理可以從持久化存儲中保存和提取資訊。
4. **個人化會隨時間改善** — 存越多偏好，代理的推薦就越準確。

### 實際應用案例
- **客服服務**：記錄客戶歷史與偏好
- **個人助理**：維持跨天或週的上下文
- **醫療保健**：追蹤病人資訊與偏好
- **電子商務**：依歷史提供個人化購物建議

### 後續步驟
- 用資料庫或向量存儲（例如 Azure AI Search）取代記憶體字典
- 為時效性資訊新增記憶到期機制
- 建立擁有共用記憶的多代理系統
- 探索 Cognee 筆記本以建立知識圖譜支持的記憶體


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件是使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於翻譯的準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始語言文件應視為權威來源。針對重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或錯誤詮釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
